# 07 · 從分析到模型：第一個預測器

前面六課我們**理解**了資料:誰容易生還、為什麼。最後一步,把理解變成**預測**——訓練一個模型,輸入乘客資料、輸出「會不會生還」。

這課是資料科學與機器學習的**交接點**:用前面清理 + 特徵工程的成果,接上 `ml/scikit-learn` 教過的 `fit` / `predict`,建一個 baseline。

## 1. 載入、清理、特徵工程(濃縮前幾課)

In [ ]:
%pip install -q -U seaborn scikit-learn

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["family_size"] = df["sibsp"] + df["parch"] + 1
features = ["pclass", "sex", "age", "fare", "family_size"]
X = df[features]
y = df["survived"]
print("特徵:", features)

## 2. 切訓練/測試集

用測試集評估,才知道模型對**沒看過的人**準不準(避免自我感覺良好)。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"訓練 {len(X_train)} 筆、測試 {len(X_test)} 筆")

## 3. 訓練一個 baseline:邏輯迴歸

分類任務最常見的起手式,簡單、可解釋。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f"測試準確率:{acc:.1%}  (基準線是『全猜死』的 {1 - y.mean():.1%})")

## 4. 模型在看什麼?看係數

邏輯迴歸的係數方向,呼應我們 EDA 的發現:女性(sex=1)、高艙等強烈正向。

In [ ]:
import pandas as pd

coef = pd.Series(model.coef_[0], index=features).sort_values()
print("特徵係數(正=提高生還機率):")
print(coef.round(3))
# sex 大正值、pclass 負值——模型自己「學到」了我們前面用 EDA 看到的故事。

## 小結

- 把清理 + 特徵工程的成果,接上 sklearn 的 `fit` / `predict`,建了第一個生還預測器。
- **訓練/測試切分**才能誠實評估;**準確率**要跟基準線比才有意義。
- 模型係數**呼應 EDA**:性別與艙等是關鍵——分析與建模互相印證。
- 想更準?回 `ml/scikit-learn` 軌道換樹模型、調參、交叉驗證。

下一課(壓軸):把整條流程串成一份**完整分析報告**。